# Association Rule Mining — Apriori Algorithm

In this module, we will use Association Rule Mining to discover relationships between products that are frequently purchased together.

We will use transaction data from a retail store.

### Problem Statement

The objective is to identify products that customers frequently purchase together and generate association rules from these relationships.

### Input

Transaction-level retail data containing:

- Invoice/Transaction ID
- Product/Item
- Quantity
- Other transaction information

### Output

Association rules such as:

> If a customer purchases Product A, they are likely to purchase Product B.

### Algorithm

**Apriori Algorithm**

### Important Concepts

- Itemset
- Frequent Itemset
- Support
- Confidence
- Lift
- Association Rule

### Workflow

Dataset → Data Exploration → Data Cleaning → Transaction Formation → One-Hot Encoding → Apriori → Frequent Itemsets → Association Rules → Support → Confidence → Lift → Rule Interpretation

## Business Problem

A retail store contains thousands of customer transactions.

Instead of analyzing customers individually, we want to discover patterns such as:

- Which products are frequently purchased together?
- Which products can be recommended together?
- Which products should be placed near each other?
- Which products can be used for cross-selling?

Association Rule Mining helps us discover these relationships automatically.

In [ ]:
!pip install -q kaggle mlxtend

In [ ]:
import os
from getpass import getpass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

print("Libraries imported successfully.")

## Kaggle API Authentication

We will download the retail transaction dataset directly from Kaggle.

The Kaggle API token will be entered securely using `getpass()`.

In [ ]:
kaggle_token = getpass("Enter your Kaggle API token: ")

os.environ["KAGGLE_API_TOKEN"] = kaggle_token

del kaggle_token

print("Kaggle API token configured successfully.")

## Download Dataset

We will use an Online Retail transaction dataset.

Each row represents a product within a transaction.

The important columns for Association Rule Mining are:

- `InvoiceNo` → transaction identifier
- `Description` → product name
- `Quantity` → quantity purchased

In [ ]:
!mkdir -p /content/online_retail_association_dataset

!kaggle datasets download \
    -d thedevastator/online-retail-sales-and-customer-data \
    -p /content/online_retail_association_dataset \
    --unzip

In [ ]:
os.listdir("/content/online_retail_association_dataset")

In [ ]:
dataset_files = [
    file
    for file in os.listdir("/content/online_retail_association_dataset")
    if file.lower().endswith((".csv", ".xlsx", ".xls"))
]

dataset_files

In [ ]:
dataset_path = os.path.join(
    "/content/online_retail_association_dataset",
    dataset_files[0]
)

if dataset_path.lower().endswith(".csv"):
    df = pd.read_csv(dataset_path)
else:
    df = pd.read_excel(dataset_path)

df.head()

## 1. Understanding the Dataset

Before applying Apriori, we need to understand the structure and quality of the transaction data.

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])print("Columns:")

for column in df.columns:
    print(column)

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

In [ ]:
missing_values = df.isnull().sum()

missing_values[missing_values > 0]

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

## 2. Data Cleaning

Association Rule Mining requires clean transaction data.

We will:

1. Remove transactions without a product description
2. Remove cancelled transactions
3. Remove transactions with zero or negative quantities
4. Remove products with invalid prices where applicable
5. Remove duplicate transaction-product rows where necessary

Cancelled transactions in the Online Retail dataset can be identified using invoice numbers beginning with `C`.

In [ ]:
print("Original shape:", df.shape)

df = df.dropna(subset=["Description"])

df = df[
    ~df["InvoiceNo"]
    .astype(str)
    .str.startswith("C")
]

df = df[df["Quantity"] > 0]

print("Cleaned shape:", df.shape)

In [ ]:
print("Missing descriptions:", df["Description"].isnull().sum())
print("Invalid quantities:", (df["Quantity"] <= 0).sum())
print(
    "Cancelled transactions:",
    df["InvoiceNo"].astype(str).str.startswith("C").sum()
)

## Why Are We Removing Cancelled Transactions?

Association Rule Mining is designed to discover products that customers actually purchased together.

A cancelled transaction does not represent a completed purchase.

Including cancelled transactions could introduce misleading relationships into our rules.

Therefore, cancelled invoices are removed.

## 3. Understanding the Transaction Structure

Apriori does not work directly with the original transaction table.

We need to transform the data into a structure where:

- Each row represents one transaction
- Each column represents one product
- The value indicates whether that product appeared in the transaction

Example:

| Transaction | Bread | Milk | Butter |
|---|---:|---:|---:|
| T1 | 1 | 1 | 0 |
| T2 | 1 | 0 | 1 |
| T3 | 1 | 1 | 1 |

This is called **one-hot encoded transaction data**.

In [ ]:
print("Number of transactions:", df["InvoiceNo"].nunique())
print("Number of unique products:", df["Description"].nunique())

In [ ]:
transaction_product = (
    df.groupby(["InvoiceNo", "Description"])["Quantity"]
    .sum()
    .unstack()
    .fillna(0)
)

transaction_product.head()

## 4. One-Hot Encoding

Apriori needs binary values.

Therefore:

- `1` → product was purchased in the transaction
- `0` → product was not purchased in the transaction

The actual quantity is not important for basic association rule mining.

We are interested in whether the product appeared in the transaction.

In [ ]:
transaction_binary = transaction_product.applymap(
    lambda x: 1 if x > 0 else 0
)

transaction_binary.head()

In [ ]:
print("Transactions:", transaction_binary.shape[0])
print("Products:", transaction_binary.shape[1])

In [ ]:
transaction_binary.sum().sort_values(
    ascending=False
).head(20)

## 5. Most Frequently Purchased Products

Before generating association rules, let's examine which products occur most frequently across transactions.

This helps us understand the transaction data and gives us an idea of which products may appear frequently in the resulting itemsets.

In [ ]:
top_products = (
    transaction_binary.sum()
    .sort_values(ascending=False)
    .head(20)
)

plt.figure(figsize=(10, 6))

sns.barplot(
    x=top_products.values,
    y=top_products.index
)

plt.xlabel("Number of Transactions")
plt.ylabel("Product")
plt.title("Top 20 Most Frequently Purchased Products")

plt.show()

# 6. Apriori Algorithm

Apriori is an algorithm used to discover **frequent itemsets** in transactional data.

The algorithm follows an important principle:

> If an itemset is frequent, all of its subsets must also be frequent.

This is called the **Apriori property**.

### Basic Process

1. Find frequent individual items
2. Generate candidate itemsets
3. Calculate their support
4. Remove itemsets below minimum support
5. Generate larger itemsets
6. Repeat the process
7. Stop when no more frequent itemsets can be generated

## 7. Support

Support tells us how frequently an itemset appears in the complete transaction dataset.

### Formula

**Support(X) = Number of transactions containing X / Total number of transactions**

For example:

If 100 out of 1,000 transactions contain Milk:

**Support(Milk) = 100 / 1,000 = 0.10**

So Milk has a support of 10%.

In [ ]:
minimum_support = 0.02

frequent_itemsets = apriori(
    transaction_binary,
    min_support=minimum_support,
    use_colnames=True
)

frequent_itemsets.head()

In [ ]:
print(
    "Number of frequent itemsets:",
    len(frequent_itemsets)
)

In [ ]:
frequent_itemsets.sort_values(
    by="support",
    ascending=False
).head(20)

## Understanding Frequent Itemsets

An itemset is a collection of one or more products.

Examples:

- `{Milk}`
- `{Bread}`
- `{Milk, Bread}`
- `{Milk, Bread, Butter}`

A **frequent itemset** is an itemset whose support is greater than or equal to the minimum support threshold.

The `min_support` parameter controls how frequently an itemset must occur to be considered frequent.

In [ ]:
frequent_itemsets["itemset_length"] = (
    frequent_itemsets["itemsets"].apply(len)
)

frequent_itemsets[
    ["support", "itemsets", "itemset_length"]
].head()

In [ ]:
frequent_itemsets["itemset_length"].value_counts().sort_index()

# 8. Generating Association Rules

Frequent itemsets tell us which products occur together frequently.

However, we also want to understand directional relationships such as:

**Product A → Product B**

For this, we generate association rules.

The major metrics are:

- Support
- Confidence
- Lift

## Confidence

Confidence measures how often Y is purchased when X is purchased.

### Formula

**Confidence(X → Y) = Support(X ∪ Y) / Support(X)**

For example:

If 60% of customers who purchase Bread also purchase Milk:

**Confidence(Bread → Milk) = 0.60**

Higher confidence means that Y occurs more frequently when X occurs.

## Lift

Lift measures how much more likely Y is to be purchased when X is purchased compared with Y being purchased independently.

### Formula

**Lift(X → Y) = Confidence(X → Y) / Support(Y)**

### Interpretation

- Lift > 1 → positive association
- Lift = 1 → no meaningful association
- Lift < 1 → negative association

A lift greater than 1 indicates that the products occur together more often than would be expected if they were independent.

In [ ]:
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.3
)

rules.head()

In [ ]:
print("Number of association rules:", len(rules))

In [ ]:
rules[
    [
        "antecedents",
        "consequents",
        "support",
        "confidence",
        "lift"
    ]
].head(20)

## 9. Filtering Strong Association Rules

A large number of rules may be generated.

Not every rule is useful.

We can filter rules using:

- Minimum confidence
- Minimum lift
- Minimum support

For example, we can focus on rules with:

- Confidence ≥ 50%
- Lift > 1

In [ ]:
strong_rules = rules[
    (rules["confidence"] >= 0.50) &
    (rules["lift"] > 1)
].copy()

print("Strong rules:", len(strong_rules))

In [ ]:
strong_rules[
    [
        "antecedents",
        "consequents",
        "support",
        "confidence",
        "lift"
    ]
].sort_values(
    by="lift",
    ascending=False
).head(20)

## Interpreting a Rule

Suppose we obtain:

**{Product A} → {Product B}**

with:

- Support = 0.05
- Confidence = 0.70
- Lift = 2.1

This means:

- The combination appears in 5% of all transactions.
- 70% of transactions containing Product A also contain Product B.
- Product B is purchased together with Product A at a rate 2.1 times higher than its baseline purchase rate.

Therefore, the rule represents a potentially useful association.

In [ ]:
top_rules = (
    strong_rules
    .sort_values(
        by=["lift", "confidence"],
        ascending=False
    )
    .head(20)
)

top_rules[
    [
        "antecedents",
        "consequents",
        "support",
        "confidence",
        "lift"
    ]
]

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=rules,
    x="support",
    y="confidence",
    size="lift",
    hue="lift",
    palette="viridis",
    sizes=(40, 300)
)

plt.title("Association Rules: Support vs Confidence")
plt.xlabel("Support")
plt.ylabel("Confidence")

plt.show()

# Part 1 Complete

In Part 1, we:

- Loaded a real-world retail transaction dataset
- Explored the dataset
- Checked missing values and duplicates
- Removed invalid transactions
- Removed cancelled transactions
- Identified transactions and products
- Converted transaction data into one-hot encoded format
- Calculated frequent itemsets using Apriori
- Generated association rules
- Calculated Support
- Calculated Confidence
- Calculated Lift
- Filtered strong association rules

### Current Workflow

Retail Dataset

↓

Data Exploration

↓

Data Cleaning

↓

Transaction Formation

↓

One-Hot Encoding

↓

Apriori Algorithm

↓

Frequent Itemsets

↓

Association Rules

↓

Support

↓

Confidence

↓

Lift

↓

Strong Rules

### Part 2

In Part 2, we will analyze the generated rules in more detail and determine which rules are most useful from a business perspective.

# Part 2 — Association Rule Evaluation & Business Interpretation

In Part 1, we generated frequent itemsets and association rules using the Apriori algorithm.

In Part 2, we will:

1. Analyze the generated rules
2. Compare Support, Confidence, and Lift
3. Identify the strongest rules
4. Filter redundant or weak rules
5. Visualize the rules
6. Examine rules with multiple products
7. Interpret the rules from a business perspective
8. Build a simple product recommendation function
9. Summarize the final findings

## 1. Reviewing the Generated Rules

An association rule has the following structure:

**Antecedent → Consequent**

For example:

**{Product A} → {Product B}**

The antecedent represents the product or products that are already present.

The consequent represents the product or products associated with them.

We will use Support, Confidence, and Lift to evaluate the usefulness of each rule.

In [ ]:
rules[
    [
        "antecedents",
        "consequents",
        "support",
        "confidence",
        "lift"
    ]
].head(10)

## 2. Understanding the Rule Metrics

### Support

Measures how frequently the complete rule occurs in the dataset.

Higher support means the relationship appears in more transactions.

### Confidence

Measures how often the consequent occurs when the antecedent occurs.

Higher confidence means the rule is more reliable.

### Lift

Measures how strongly the antecedent and consequent are associated compared with random occurrence.

- Lift > 1 → positive association
- Lift = 1 → no association
- Lift < 1 → negative association

For finding useful relationships, we generally look for rules with reasonable support, high confidence, and lift greater than 1.

## 3. Rules with the Highest Lift

Lift is useful for identifying relationships where two products occur together much more often than expected.

Let's identify the rules with the highest lift.

In [ ]:
highest_lift_rules = (
    rules
    .sort_values(
        by="lift",
        ascending=False
    )
    .head(20)
)

highest_lift_rules[
    [
        "antecedents",
        "consequents",
        "support",
        "confidence",
        "lift"
    ]
]

### Important Observation

A rule with extremely high lift is not automatically the most useful business rule.

For example, a rule may have:

- Very high lift
- Very high confidence
- But extremely low support

This means the relationship is strong but occurs in very few transactions.

Therefore, we should consider all three metrics together.

## 4. Rules with the Highest Confidence

Confidence tells us how frequently the consequent occurs when the antecedent occurs.

Let's identify the rules with the highest confidence.

In [ ]:
highest_confidence_rules = (
    rules
    .sort_values(
        by="confidence",
        ascending=False
    )
    .head(20)
)

highest_confidence_rules[
    [
        "antecedents",
        "consequents",
        "support",
        "confidence",
        "lift"
    ]
]

## Confidence vs Lift

Confidence and lift answer different questions.

### Confidence

"If a customer buys X, how often do they also buy Y?"

### Lift

"How much more likely is Y when X is purchased compared with Y's normal purchase rate?"

Therefore, a high-confidence rule can still have a relatively low lift if the consequent product is already extremely common.

## 5. Creating a More Useful Rule Set

We can create stricter thresholds to focus on stronger relationships.

For this analysis, we will use:

- Support ≥ 2%
- Confidence ≥ 50%
- Lift > 1

These thresholds can be adjusted depending on the dataset and business requirements.

In [ ]:
filtered_rules = rules[
    (rules["support"] >= 0.02) &
    (rules["confidence"] >= 0.50) &
    (rules["lift"] > 1)
].copy()

print("Original rules:", len(rules))
print("Filtered rules:", len(filtered_rules))

In [ ]:
filtered_rules[
    [
        "antecedents",
        "consequents",
        "support",
        "confidence",
        "lift"
    ]
].sort_values(
    by=["lift", "confidence"],
    ascending=False
).head(20)

## 6. Analyzing Rule Complexity

Association rules can contain:

- One product in the antecedent
- Multiple products in the antecedent
- One or multiple products in the consequent

Examples:

**{A} → {B}**

**{A, B} → {C}**

**{A, B} → {C, D}**

Let's examine the size of the antecedent and consequent.

In [ ]:
filtered_rules["antecedent_length"] = (
    filtered_rules["antecedents"].apply(len)
)

filtered_rules["consequent_length"] = (
    filtered_rules["consequents"].apply(len)
)

filtered_rules[
    [
        "antecedents",
        "consequents",
        "antecedent_length",
        "consequent_length"
    ]
].head(10)

In [ ]:
filtered_rules[
    [
        "antecedents",
        "consequents",
        "support",
        "confidence",
        "lift",
        "antecedent_length",
        "consequent_length"
    ]
].sort_values(
    by="lift",
    ascending=False
).head(20)

## 7. Single-Product Association Rules

For practical product recommendation, simple rules are often easier to interpret.

We will focus on rules where:

- The antecedent contains one product
- The consequent contains one product

In [ ]:
single_product_rules = filtered_rules[
    (filtered_rules["antecedent_length"] == 1) &
    (filtered_rules["consequent_length"] == 1)
].copy()

print(
    "Single-product rules:",
    len(single_product_rules)
)

In [ ]:
single_product_rules[
    [
        "antecedents",
        "consequents",
        "support",
        "confidence",
        "lift"
    ]
].sort_values(
    by="lift",
    ascending=False
).head(20)

## 8. Visualizing Association Rules

A scatter plot can help us understand the relationship between:

- Support
- Confidence
- Lift

Each point represents an association rule.

Large lift values indicate stronger associations.

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=filtered_rules,
    x="support",
    y="confidence",
    size="lift",
    hue="lift",
    palette="viridis",
    sizes=(50, 400)
)

plt.xlabel("Support")
plt.ylabel("Confidence")
plt.title("Strong Association Rules")

plt.show()

## 9. Finding the Most Useful Rules

There is no single metric that completely describes the usefulness of a rule.

For this analysis, we will prioritize:

1. Reasonable support
2. High confidence
3. High lift

Let's examine rules that perform well across these measures.

In [ ]:
top_business_rules = (
    filtered_rules
    .sort_values(
        by=["lift", "confidence", "support"],
        ascending=False
    )
    .head(20)
)

top_business_rules[
    [
        "antecedents",
        "consequents",
        "support",
        "confidence",
        "lift"
    ]
]

## 10. Making Rules Easier to Read

The antecedent and consequent are stored as Python `frozenset` objects.

We can convert them into readable text so that the rules can be presented more easily.

In [ ]:
def format_itemset(itemset):
    return ", ".join(sorted(itemset))


readable_rules = top_business_rules.copy()

readable_rules["Rule"] = (
    readable_rules["antecedents"].apply(format_itemset)
    + "  →  "
    + readable_rules["consequents"].apply(format_itemset)
)

readable_rules[
    [
        "Rule",
        "support",
        "confidence",
        "lift"
    ]
]

## 11. Building a Simple Product Recommendation System

Association rules can be used to make basic product recommendations.

For example:

If a customer has purchased Product A:

**Product A → Product B**

then Product B can be recommended.

We will create a simple function that searches the association rules for a given product.

In [ ]:
def recommend_products(product_name, rules_df, top_n=5):
    recommendations = rules_df[
        rules_df["antecedents"].apply(
            lambda items: product_name in items
        )
    ].copy()

    recommendations = recommendations.sort_values(
        by=["lift", "confidence"],
        ascending=False
    )

    return recommendations[
        [
            "antecedents",
            "consequents",
            "support",
            "confidence",
            "lift"
        ]
    ].head(top_n)

## 12. Test the Recommendation System

Enter the exact product name from the dataset.

In [ ]:
available_products = sorted(
    single_product_rules["antecedents"]
    .explode()
    .unique()
)

available_products[:20]

In [ ]:
product_name = available_products[0]

recommend_products(
    product_name,
    single_product_rules,
    top_n=5
)

### Recommendation Interpretation

The function searches for rules where the selected product appears in the antecedent.

The consequent products are potential recommendations.

The recommendations are ranked using:

1. Lift
2. Confidence

These recommendations represent associations discovered from historical transaction data.

They should not be interpreted as guaranteed purchases.

## 13. Business Applications of Association Rules

Association Rule Mining can be used in several areas.

### Market Basket Analysis

Identify products that customers frequently purchase together.

### Product Recommendations

Recommend related products based on previous purchases.

### Cross-Selling

Suggest additional products to customers.

### Store Layout

Products with strong associations can potentially be positioned closer together.

### Promotional Bundles

Frequently associated products can be combined into offers or packages.

### E-Commerce

Association rules can support "Customers who bought this also bought..." recommendations.

In [ ]:
print("Total frequent itemsets:", len(frequent_itemsets))
print("Total association rules:", len(rules))
print("Strong association rules:", len(filtered_rules))
print("Single-product rules:", len(single_product_rules))

## 14. Final Association Rule Analysis

The Apriori algorithm has allowed us to move from raw transaction data to meaningful product relationships.

### We discovered:

- Frequent product combinations
- Product-to-product associations
- Strong association rules
- High-confidence relationships
- High-lift relationships
- Potential product recommendations

The most useful rules should have sufficient support, strong confidence, and lift greater than 1.

# Association Rule Mining — Final Summary

## What We Covered

### Data Preparation
- Loaded retail transaction data
- Explored the dataset
- Checked missing values and duplicates
- Removed cancelled transactions
- Removed invalid quantities
- Prepared transaction-level data

### Transaction Representation
- Grouped products by transaction
- Converted transactions into a product matrix
- Applied one-hot encoding

### Apriori Algorithm
- Generated candidate itemsets
- Identified frequent itemsets
- Applied minimum support
- Used the Apriori property

### Association Rules
- Generated rules from frequent itemsets
- Calculated Support
- Calculated Confidence
- Calculated Lift
- Filtered strong rules

### Rule Analysis
- Compared high-support rules
- Compared high-confidence rules
- Compared high-lift rules
- Analyzed rule complexity
- Focused on single-product associations

### Visualization
- Visualized support vs confidence
- Used lift to represent association strength

### Recommendation
- Built a simple product recommendation function
- Generated product recommendations from association rules

---

## Complete Workflow

**Retail Transactions**

↓

**Data Exploration**

↓

**Data Cleaning**

↓

**Transaction Formation**

↓

**One-Hot Encoding**

↓

**Frequent Itemset Generation**

↓

**Apriori Algorithm**

↓

**Frequent Itemsets**

↓

**Association Rules**

↓

**Support**

↓

**Confidence**

↓

**Lift**

↓

**Rule Filtering**

↓

**Rule Analysis**

↓

**Product Recommendations**

↓

**Business Interpretation**

---

## Key Concepts Learned

- Association Rule Mining
- Apriori Algorithm
- Transaction Data
- Itemsets
- Frequent Itemsets
- Apriori Property
- Support
- Confidence
- Lift
- Market Basket Analysis
- Cross-Selling
- Product Recommendation

# Association Rule Mining Module Complete

We successfully used the **Apriori Algorithm** to discover relationships between products in retail transactions.

The final objective was not simply to generate rules, but to identify rules that can potentially provide useful business insights.

### Final Concepts

**Support** → How frequently the combination occurs

**Confidence** → How often the consequent occurs when the antecedent occurs

**Lift** → How strongly the two itemsets are associated compared with independent occurrence

**Apriori** → Efficiently discovers frequent itemsets using the Apriori property

**Association Rules** → Convert frequent itemsets into actionable relationships

**Market Basket Analysis** → Uses these relationships to understand purchasing behavior